In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import requests
import time
import json
from dotenv import load_dotenv

Download data for the top 2500 stocks by SEC market cap. Data last updated on: Jan 13 2025

In [5]:
# This script systematically exhausts all API keys in the .env file and sorts the downloaded data
load_dotenv()
apikeylist = os.getenv('apikeylist').split()
apikey = os.getenv("apikey")

headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()

# For non-growth-adjusted figures: [['balance-sheet-statement', 'balance'], ['cash-flow-statement', 'cashflow'], ['income-statement', 'income'], ['key-metrics', 'keymetrics']]
for pair in [['balance-sheet-statement-growth', 'balancegrowth'], ['cash-flow-statement-growth', 'cashflowgrowth'], ['income-statement-growth', 'incomegrowth']]:
    index = 0
    
    for apikey in apikeylist:
        while index < 2500:
            try:
                url = f'https://financialmodelingprep.com/api/v3/{pair[0]}/{tickers["ticker"].iloc[index]}?period=annual&apikey={apikey}'
                
                request = requests.get(url).json()
                data = pd.DataFrame(request)
                data.to_csv(f'data/{pair[1]}/{tickers["ticker"].iloc[index]}.csv')
                index += 1
            except:
                print(f"API key failed on row {index}, trying with next key")
                break

print(f"All API keys exhausted, finished on row {index}")

API key failed on row 250, trying with next key
API key failed on row 500, trying with next key
API key failed on row 750, trying with next key
API key failed on row 1000, trying with next key
API key failed on row 1250, trying with next key
API key failed on row 1500, trying with next key
API key failed on row 1750, trying with next key
API key failed on row 2000, trying with next key
API key failed on row 2250, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 0, trying with next key
API key failed on row 250, trying with next key
API key failed on row 500, trying with next key
API ke

## Preprocessing
Start by compiling the data by combining sections and removing redundant metadata

TL;DR: of the 2500 stocks, 40 stocks had missing data, 4 were removed because their metrics were compiled from the wrong fiscal year, and 3 were delisted, leaving us with 2453 stocks.


Number of metrics in each category:
- cashflow: 27
- keymetrics: 57
- balancesheet: 39
- income: 26
- total: 149

The growth datasets from financial modeling prep were used to get the percent change annually for each metric. Because the data is in a time series, it cannot be normalized without inadvertently exposing the model to future information, even if it is as small as a normalization factor. Thus, I chose to use percent change instead of raw values because the raw values can have drastically different magnitudes, allowing larger values to be more prominent in regression. Using annual percent change puts every value between -1 and 1 without exposing the model to future data.

In [102]:
# Compiles data for each stock in the list provided by the SEC
headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()
missing = 0

for i in range(2500): # Max is 10175
    ticker = tickers['ticker'].iloc[i]
    
    try:
        balancesheet = pd.read_csv(f"data/balancegrowth/{ticker}.csv", index_col=0).drop(['date', 'symbol', 'calendarYear', 'period'], axis=1)
        cashflow = pd.read_csv(f"data/cashflowgrowth/{ticker}.csv", index_col=0).drop(['date', 'symbol', 'calendarYear', 'period'], axis=1)
        keymetrics = pd.read_csv(f"data/keymetrics/{ticker}.csv", index_col=0).drop(["symbol", 'calendarYear', 'period', 'date'], axis=1)
        income = pd.read_csv(f"data/incomegrowth/{ticker}.csv", index_col=0)
        filing_date_col = pd.read_csv(f"data/income/{ticker}.csv")["fillingDate"]
        
        data = pd.concat([filing_date_col, keymetrics, balancesheet, cashflow, income], axis=1)
        data.set_index("fillingDate", inplace=True)
        data.to_csv(f'data/compiled/{ticker}.csv')
    
    except:
        print(f'Missing data for ticker {ticker}')
        missing += 1
print(missing)

Missing data for ticker SPY
Missing data for ticker RCIT
Missing data for ticker QQQ
Missing data for ticker GLD
Missing data for ticker BMWKY
Missing data for ticker IAU
Missing data for ticker DIA
Missing data for ticker MDY
Missing data for ticker ONC
Missing data for ticker BNT
Missing data for ticker USO
Missing data for ticker RADX
Missing data for ticker TTAN
Missing data for ticker SARO
Missing data for ticker FUPBY
Missing data for ticker SOBO
Missing data for ticker PONY
Missing data for ticker SNRE
Missing data for ticker JBTM
Missing data for ticker FLG
Missing data for ticker SHZNY
Missing data for ticker CURB
Missing data for ticker BKV
Missing data for ticker NDOI
Missing data for ticker VIR
Missing data for ticker LUCK
Missing data for ticker SEI
Missing data for ticker TICA
Missing data for ticker GRDN
Missing data for ticker AAPI
Missing data for ticker AGLY
Missing data for ticker IMKTA
Missing data for ticker HCI
Missing data for ticker BHLB
Missing data for ticker 

Process the data for the individual metric types

In [119]:
headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()

for i in range(2500):
    ticker = tickers['ticker'].iloc[i]
    
    for folder in ['balancegrowth', 'cashflowgrowth', 'incomegrowth', 'keymetrics']:
        try:
            data = pd.read_csv(f"data/{folder}/{ticker}.csv", index_col=0)
            filing_date_col = pd.read_csv(f"data/income/{ticker}.csv")["fillingDate"]

            data = pd.concat([filing_date_col, data], axis=1)
            data.set_index("fillingDate", inplace=True)
            data.to_csv(f'data/{folder}/{ticker}.csv')

        except:
            continue

Ensure the data is in chronological order

In [103]:
headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()

for i in range(2500):
    ticker = tickers['ticker'].iloc[i]
    
    try:
        compiled = pd.read_csv(f'data/compiled/{ticker}.csv')
        
        for i in range(len(compiled)):
            if compiled['date'].loc[i] > compiled['fillingDate'].loc[i]:
                print(ticker, compiled['date'].loc[i], compiled['fillingDate'].loc[i])
                os.remove(f'data/compiled/{ticker}.csv')
                break
    except:
        print(f'Missing data for ticker {ticker}')

Missing data for ticker SPY
Missing data for ticker RCIT
Missing data for ticker QQQ
Missing data for ticker GLD
Missing data for ticker BMWKY
Missing data for ticker IAU
Missing data for ticker DIA
Missing data for ticker MDY
RYAAY 2022-03-31 2021-03-31
Missing data for ticker ONC
Missing data for ticker BNT
Missing data for ticker USO
Missing data for ticker RADX
Missing data for ticker TTAN
Missing data for ticker SARO
Missing data for ticker FUPBY
Missing data for ticker SOBO
Missing data for ticker PONY
Missing data for ticker SNRE
Missing data for ticker JBTM
Missing data for ticker FLG
Missing data for ticker SHZNY
NVG 2024-10-31 2023-10-31
Missing data for ticker CURB
HUT 2023-06-30 2022-12-31
Missing data for ticker BKV
Missing data for ticker NDOI
Missing data for ticker VIR
Missing data for ticker LUCK
Missing data for ticker SEI
Missing data for ticker TICA
Missing data for ticker GRDN
Missing data for ticker AAPI
Missing data for ticker AGLY
NMZ 2024-10-31 2023-10-31
Missi

Code to remove duplicate columns

In [104]:
# Remove duplicate columns if there are any
for file in os.listdir("data/compiled"):
    if file != '.DS_Store':
        data = pd.read_csv(f"data/compiled/{file}", index_col=0)
        for column in ['growthInventory.1', 'growthDepreciationAndAmortization.1', 'growthNetIncome.1']:
            if column in data.columns:
                data = data.drop([column], axis=1)
        data.to_csv(f"data/compiled/{file}")

Finally, download historical prices from yfinance

In [109]:
import yfinance as yf
headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()

for i in range(0, 3000):
    ticker = tickers['ticker'].iloc[i]
    
    try:
        hist = yf.Ticker(ticker).history(period="max")['Open']
        hist.to_csv(f'data/yfinance/{ticker}.csv')
        
    except:
        print(f'Missing yfinance data for {ticker}')

$VG: possibly delisted; no timezone found
$PTCT: possibly delisted; no price data found  (1d 1926-02-08 -> 2025-01-14)
$AYR: possibly delisted; no timezone found
$JAG: possibly delisted; no timezone found


This is the makeup of the metrics used. The largest category is key metrics, which is a calculated category compiled from other metrics. Balance sheet is the largest non-calculated section. There are a total of 149 metrics.

In [118]:
ticker = 'AAPL'
cashflow = pd.read_csv(f"data/cashflowgrowth/{ticker}.csv", index_col=0).drop(['date', 'symbol', 'calendarYear', 'period', 'growthDepreciationAndAmortization', 'growthInventory', 'growthNetIncome'], axis=1)
keymetrics = pd.read_csv(f"data/keymetrics/{ticker}.csv", index_col=0).drop(['date', 'symbol', 'calendarYear', 'period'], axis=1)
balancesheet = pd.read_csv(f"data/balancegrowth/{ticker}.csv", index_col=0).drop(['date', 'symbol', 'calendarYear', 'period'], axis=1)
income = pd.read_csv(f"data/incomegrowth/{ticker}.csv", index_col=0).drop(['date', 'symbol', 'calendarYear', 'period'], axis=1)
compiled = pd.read_csv(f"data/compiled/{ticker}.csv", index_col=0).drop(['date', 'symbol', 'calendarYear', 'period'], axis=1)

print(f'cashflow: {len(cashflow.columns)}')
print(f'keymetrics: {len(keymetrics.columns)}')
print(f'balancesheet: {len(balancesheet.columns)}')
print(f'income: {len(income.columns)}')
print(f'total: {len(compiled.columns)}')

cashflow: 27
keymetrics: 57
balancesheet: 39
income: 26
total: 149


Code for getting financial data from QuickFS

In [ ]:
load_dotenv()
apikeylist = os.getenv('quickfsapikeylist').split()

headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()
i = 1

for key in apikeylist:
    while i < 2500:
        ticker = tickers['ticker'].iloc[i]
        print(i, ticker)

        try:
            r = requests.get(f"https://public-api.quickfs.net/v1/data/all-data/{ticker}:US?api_key={key}").json()
            if "error" in r.keys() and r['status'] == 429: 
                print(f"API key failed on number {i}, ticker {ticker}. Trying with next key")
                break
            elif "errors" in r.keys() and r['errors']['status'] == 404:
                print(f"Missing data for number {i}, ticker {ticker}")
                i += 1
            else:
                data = pd.DataFrame(r['data']['financials']['annual'])
                data.set_index("original_filing_date", inplace=True)
                data.to_csv(f'data/quickfs/{ticker}.csv')
                i += 1
            
        except:
            print(f"Bad data for number {i}, ticker {ticker}")
            i += 1
            
print(f"All API keys exhausted, finished on number {i}")

1 NVDA
Bad data for number 1, ticker NVDA
2 AAPL
Bad data for number 2, ticker AAPL
3 AMZN
Bad data for number 3, ticker AMZN
4 GOOGL
Bad data for number 4, ticker GOOGL
5 META
Bad data for number 5, ticker META
6 AVGO
Bad data for number 6, ticker AVGO
7 TSLA
Bad data for number 7, ticker TSLA
8 BRK-B
Missing data for number 8, ticker BRK-B
9 WMT
Bad data for number 9, ticker WMT
10 JPM
Bad data for number 10, ticker JPM
11 V
Bad data for number 11, ticker V
12 LLY
Bad data for number 12, ticker LLY
13 SPY
Missing data for number 13, ticker SPY
14 MA
Bad data for number 14, ticker MA
15 NFLX
Bad data for number 15, ticker NFLX
16 ORCL
Bad data for number 16, ticker ORCL
17 COST
Bad data for number 17, ticker COST
18 XOM
Bad data for number 18, ticker XOM
19 PG
Bad data for number 19, ticker PG
20 JNJ
Bad data for number 20, ticker JNJ
21 HD
Bad data for number 21, ticker HD
22 SAP
Bad data for number 22, ticker SAP
23 BAC
Bad data for number 23, ticker BAC
24 ABBV
Bad data for number 

Of the 2500 stocks, 42 stocks had no data on quickfs and one stock had data that could not be interpreted, leaving me with 2457.

In [11]:
len(os.listdir("data/quickfs"))

2457

Find which stocks are not included in the database

In [14]:
headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()

api_key = "d83b898c0fa5b15f8cc7455b3026f6e48a4f7443"
r = requests.get(f"https://public-api.quickfs.net/v1/companies/US?api_key={api_key}").json()
number = 0

for i in range(2500):
    if f'{tickers["ticker"].iloc[i]}:US' not in r['data']:
        print(i, tickers['ticker'].iloc[i])
        number += 1
print(f'Missing {number} stocks')

8 BRK-B
13 SPY
45 QQQ
220 GLD
278 IAU
292 CTA-PA
388 DIA
410 MDY
510 EXE
573 ONC
671 BF-A
824 PRMB
834 RADX
930 BNT
934 USO
945 SLV
979 BPYPP
988 TTAN
1079 NBIS
1158 MOG-A
1177 OS
1184 CMS-PB
1188 UELMO
1288 AMTM
1338 PONY
1366 TLX
1416 INGM
1461 SNRE
1535 JBTM
1537 FAURY
1566 FLG
1655 ECG
1740 GRP-UN
1829 AKO-A
1880 DBC
2070 NDOI
2102 LGF-A
2121 AILIH
2263 LUCK
2304 TICA
2324 GSG
2441 AGLY
Missing 42 stocks


Remove rows that do not contain filing dates

In [43]:
for file in os.listdir("data/quickfs"):
    data = pd.read_csv(f"data/quickfs/{file}", index_col=0)
    if '-' in data.index:
        data = data.drop("-")
    data.to_csv(f"data/quickfs/{file}")

Download the list of metrics so we can categorize the data

In [88]:
api_key = "d83b898c0fa5b15f8cc7455b3026f6e48a4f7443"

r = requests.get(f"https://public-api.quickfs.net/v1/metrics?api_key={api_key}")

my_data = r.json()
metric_data = pd.DataFrame(my_data['data'])
# metric_data.to_csv("data/metric_metadata.csv")

In [61]:
number = 0
for metric in my_data['data']:
    if 'normal' in metric['company_types']:
        print(metric['metric'], metric['statement_type'])
        number += 1
print(number)

revenue income_statement
cogs income_statement
gross_profit income_statement
sga income_statement
rnd income_statement
special_charges income_statement
other_opex income_statement
total_opex income_statement
operating_income income_statement
net_interest_income_normal income_statement
other_nonoperating_income income_statement
pretax_income income_statement
income_tax income_statement
net_income_continuing income_statement
interest_income income_statement
interest_expense income_statement
net_income_discontinued income_statement
income_allocated_to_minority_interest income_statement
other_income_statement_items income_statement
net_income income_statement
preferred_dividends income_statement
net_income_available_to_shareholders income_statement
eps_basic income_statement
eps_diluted income_statement
shares_basic income_statement
shares_diluted income_statement
cash_and_equiv balance_sheet
st_investments balance_sheet
receivables balance_sheet
inventories balance_sheet
other_current_ass

Because investment funds and banks do not report all of the information that most publicly traded companies do, it is better to consider them separately. As such, I will not include the 359 banking and investment fund stocks.

In [70]:
number = 0
for file in os.listdir("data/quickfs"):
    data = pd.read_csv(f"data/quickfs/{file}", index_col=0)
    if len(data.columns) != 195:
        print(file, len(data.columns))
        os.rename(f"data/quickfs/{file}", f"data/quickfsbanking/{file}")
        number += 1
print(number)

0


Check that every single file contains all of the same columns

In [79]:
missing_cols = ['long_term_debt_and_capital_lease_obligation', 'original_filing_date', 'restated_filing_date', 'preliminary']
for file in os.listdir("data/quickfs"):
    data = pd.read_csv(f"data/quickfs/{file}", index_col=0)
    for metric in my_data['data']:
        if 'normal' in metric['company_types']:
            if metric['metric'] not in data.columns and metric['metric'] not in missing_cols:
                print(file, metric['metric'], metric['statement_type'])

Remove duplicate filing dates. Some quarterly reports for different quarters are filed on the same day, so we will lose some data. However, it is more important that the filing dates can be used as timestamps as they are realistic to when data would become accessible.

In [74]:
for file in os.listdir("data/quickfs"):
    data = pd.read_csv(f"data/quickfs/{file}", index_col=0)
    data = data.loc[~data.index.duplicated(keep='last')]
    data.to_csv(f"data/quickfs/{file}")

Finally, sort the data based on metric type

In [2]:
data = pd.read_csv(f"data/quickfs/AAPL.csv", index_col=0)
new_df = pd.DataFrame()
new_df.index = data.index
metric_type = 'income_statement' # income_statement, balance_sheet, cash_flow_statement, computed

for metric in my_data['data']:
    if 'normal' in metric['company_types'] and metric['metric'] not in missing_cols and metric['statement_type'] == metric_type:
        new_df[metric['metric']] = data[metric['metric']]
new_df

NameError: name 'my_data' is not defined

In [94]:
from quickfsfunctions import *
# all, income_statement, balance_sheet, cash_flow_statement, computed
len(preprocess("AAPL.csv", "2002-10-01", 1, "computed")[0])

83

In [16]:
year_list = pd.Series()


for file in os.listdir("data/quickfs"):
    data = pd.read_csv(f"data/quickfs/{file}", index_col=0)
    year_list.loc[len(year_list)] = int(data.index[-1][:4]) - int(data.index[0][:4])

'''
headers = {'User-Agent': "alexwycoff18@gmail.com"}
tickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()['ticker']
banking = 0

for i in range(500):
    ticker = tickers.iloc[i]
    if f"{ticker}.csv" in os.listdir("data/quickfs"):
        data = pd.read_csv(f"data/quickfs/{ticker}.csv", index_col=0)
        year_list.loc[len(year_list)] = int(data.index[-1][:4]) - int(data.index[0][:4])
    elif f"{ticker}.csv" in os.listdir("data/quickfsbanking"):
        banking += 1
print(banking)
'''

'\nheaders = {\'User-Agent\': "alexwycoff18@gmail.com"}\ntickers = pd.DataFrame(requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers).json()).transpose()[\'ticker\']\nbanking = 0\n\nfor i in range(500):\n    ticker = tickers.iloc[i]\n    if f"{ticker}.csv" in os.listdir("data/quickfs"):\n        data = pd.read_csv(f"data/quickfs/{ticker}.csv", index_col=0)\n        year_list.loc[len(year_list)] = int(data.index[-1][:4]) - int(data.index[0][:4])\n    elif f"{ticker}.csv" in os.listdir("data/quickfsbanking"):\n        banking += 1\nprint(banking)\n'

We can also find out how many stocks have certain amounts of years of filing data.

In [17]:
table = pd.DataFrame()
table["Years of Data"] = ['< 1 Year', '1-5 Years', '6-10 Years', '11-15 Years', '16-20 Years', '> 20 Years', 'Total']
table['Number of Stocks'] = [len(year_list[year_list < 1]), len(year_list[(year_list > 0) & (year_list < 6)]), len(year_list[(year_list > 5) & (year_list < 11)]), len(year_list[(year_list > 10) & (year_list < 16)]), len(year_list[(year_list > 15) & (year_list < 21)]), len(year_list[year_list > 20]), len(year_list)]
table

,Years of Data,Number of Stocks
0,< 1 Year,20
1,1-5 Years,358
2,6-10 Years,264
3,11-15 Years,228
4,16-20 Years,249
5,> 20 Years,979
6,Total,2098


Since more than half of the total stocks have more than 20 years of data, I chose to train the stock in the interval from 2003 to 2012 and test it from 2013 to 2023.